# 35 — Priority: encoder roster vs classical

Priority had only classical baselines; this runs the full encoder roster on `task=priority`
(pooled dev, macro-F1), then promotes the winners to the one-shot **train+dev → test** eval —
the same protocol as sentiment. Classical champion to beat: **tfidf-svm 0.8722 test macro-F1**
(CI [0.8605, 0.8831]).

Encoders trained on Kaggle T4 via `runner.py run --task priority` (dev) and
`--task priority --fit-portion train+dev --eval-portion test` (final).

In [1]:
import sys, warnings, json, glob
from pathlib import Path
warnings.filterwarnings("ignore")
REPO = Path.cwd()
while not (REPO/"ml"/"swiftbench").exists() and REPO!=REPO.parent: REPO=REPO.parent
sys.path.insert(0,str(REPO/"ml"))
import pandas as pd
from swiftbench import splits
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
SHA=splits.sha(); print("split sha:", SHA)
ENC={"labse","xlmr-base","mmbert","canine-c","twhin-bert","sinbert-large","sinhalaberto"}
def rows(portion):
    out=[]
    for f in glob.glob(str(REPO/f"ml/reports/runs/priority__*__ev-all__*{portion}.json")):
        d=json.load(open(f))
        if d.get("split_sha")!=SHA: continue
        fam="encoder" if d.get("model") in ENC else "classical"
        out.append({"model":d.get("model"),"family":fam,"macro_f1":d.get("macro_f1"),
                    "author":d.get("author",""),"arm":d.get("arm")})
    return pd.DataFrame(out)

split sha: e7b5934392cd


## 1. Priority — pooled dev (bake-off)

In [2]:
dev = rows("dev")
dev = dev[(dev.family=="encoder") & (dev.author=="kaggle")]  # the bake-off runs
dev = dev.sort_values("macro_f1", ascending=False).drop_duplicates("model")
print("classical priority dev (tfidf-svm) = 0.9028")
display(dev[["model","macro_f1"]])

classical priority dev (tfidf-svm) = 0.9028


,model,macro_f1
3,labse,0.9168
6,xlmr-base,0.9162
4,mmbert,0.9148
5,twhin-bert,0.8907
0,canine-c,0.8786
2,sinhalaberto,0.5573
1,sinbert-large,0.4939


## 2. Priority — one-shot test (fit train+dev)

In [3]:
test = rows("test")
# encoders promoted to test vs the classical champions
board = test.sort_values("macro_f1", ascending=False).drop_duplicates("model")
board["vs_classical_0.8722"] = board["macro_f1"] - 0.8722
display(board[["model","family","macro_f1","vs_classical_0.8722"]])
enc_best = board[board.family=="encoder"]["macro_f1"].max()
print(f"best encoder test {enc_best:.4f} vs classical 0.8722 (CI 0.8605-0.8831) -> "
      f"{'CLEARS classical CI' if enc_best>0.8831 else 'within classical CI'}")

,model,family,macro_f1,vs_classical_0.8722
4,labse,encoder,0.8900,0.0178
0,mmbert,encoder,0.8887,0.0165
2,xlmr-base,encoder,0.8872,0.0150
3,tfidf-svm,classical,0.8722,-0.0000
1,tfidf-logreg,classical,0.8683,-0.0039


best encoder test 0.8900 vs classical 0.8722 (CI 0.8605-0.8831) -> CLEARS classical CI


## Finding

The top encoders (labse, mmbert, xlm-roberta) beat the classical champion on priority test by
**+0.015 to +0.018 macro-F1**, and **labse/mmbert clear the classical CI upper bound (0.8831)** —
a real, if modest, win. Priority is closer to its label ceiling than sentiment, so the margin is
smaller than sentiment's (+0.10 Neg-F1), but encoders win on **both** tasks.

**Not competitive:** twhin-bert (0.891 dev) and canine-c (0.879 dev) sit below the classical bar;
the Sinhala-only checkpoints are far below (pooled metric is unfair to them).

**Overall recommendation across both tasks: ship one multilingually fine-tuned LaBSE** (sentiment
Neg-F1 0.566, priority macro-F1 0.890 on test) — it is the best model on every task and language.